<a href="https://colab.research.google.com/github/srivastava071/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%pip -q install duckdb huggingface_hub

In [ ]:
import os
import getpass

HF_TOKEN = os.environ.get("HF_TOKEN")

if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass

HF_TOKEN = HF_TOKEN or getpass.getpass(
    "Paste your Hugging Face READ token (hf_...): "
)

print("HF token found:", bool(HF_TOKEN))

Paste your Hugging Face READ token (hf_...): ··········
HF token found: True


In [ ]:
import duckdb

con = duckdb.connect()

con.execute(
    f"CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

print("DuckDB connected.")

DuckDB connected.


In [ ]:
REL = "hf://datasets/FlyRank/internship-warehouse"

TABLES = {
    "fact_daily": f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    "dim_content": f"read_parquet('{REL}/dim_content.parquet')",
}

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/srivastava071/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

I will create a simple baseline score to prioritize pages for human review. The score will give more priority to pages with meaningful search visibility, older content, and a reasonable search position. These signals will be combined rather than treating any one signal as proof that a page needs action.

The output will be used as decision-support for a content team, not as an automatic instruction to change a page.

Reason codes:
- STALE_VISIBLE: the page is older and has meaningful search visibility.
- VISIBLE_OPPORTUNITY: the page has meaningful search visibility and a search position where review may be useful.
- REVIEW: the page has some measurable signals but does not strongly match the other conditions.

Actions:
- REFRESH_REVIEW: review whether the content needs updating.
- OPPORTUNITY_REVIEW: review the page for improvement opportunities.
- GENERAL_REVIEW: keep the page in the review queue for human assessment.

In [ ]:
# Define the reason codes and actions used by the baseline rule

reason_codes = {
    "STALE_VISIBLE": "Older content with meaningful search visibility",
    "VISIBLE_OPPORTUNITY": "Meaningful visibility with a potentially useful search position",
    "REVIEW": "Measurable signals suggest the page may be worth human review"
}

actions = {
    "STALE_VISIBLE": "REFRESH_REVIEW",
    "VISIBLE_OPPORTUNITY": "OPPORTUNITY_REVIEW",
    "REVIEW": "GENERAL_REVIEW"
}

print("Reason codes:")
for code, meaning in reason_codes.items():
    print(f"{code}: {meaning}")

print("\nActions:")
for code, action in actions.items():
    print(f"{code}: {action}")

Reason codes:
STALE_VISIBLE: Older content with meaningful search visibility
VISIBLE_OPPORTUNITY: Meaningful visibility with a potentially useful search position
REVIEW: Measurable signals suggest the page may be worth human review

Actions:
STALE_VISIBLE: REFRESH_REVIEW
VISIBLE_OPPORTUNITY: OPPORTUNITY_REVIEW
REVIEW: GENERAL_REVIEW


In [ ]:
TABLES["fact_query_90d"] = (
    f"read_parquet('{REL}/fact_content_query_90d.parquet')"
)

print(TABLES.keys())

dict_keys(['fact_daily', 'dim_content', 'fact_query_90d'])


In [ ]:
con.sql(f"""
SELECT COUNT(*) AS rows
FROM {TABLES['fact_query_90d']}
""").df()

,rows
0,2414248


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [ ]:
# Build the baseline feature table for the March decision point

baseline_data = con.sql(f"""
WITH previous_window AS (
    SELECT
        f.client_hash_id,
        f.content_hash_id,

        SUM(f.gsc_impressions) AS previous_30d_impressions,
        SUM(f.gsc_clicks) AS previous_30d_clicks,
        AVG(f.gsc_avg_position) AS previous_30d_avg_position

    FROM {TABLES['fact_daily']} f

    WHERE f.report_date >= DATE '2026-02-01'
      AND f.report_date < DATE '2026-03-01'
      AND f.gsc_data_available IS TRUE

    GROUP BY
        f.client_hash_id,
        f.content_hash_id
),

query_signals AS (
    SELECT
        content_hash_id,
        ANY_VALUE(content_visible_query_count) AS visible_query_count,
        MAX(impressions_90d) / NULLIF(SUM(impressions_90d), 0) AS top_query_share

    FROM {TABLES['fact_query_90d']}

    GROUP BY content_hash_id
),

content_age AS (
    SELECT
        content_hash_id,
        DATE_DIFF(
            'day',
            CAST(content_created_date AS DATE),
            DATE '2026-03-01'
        ) AS age_days

    FROM {TABLES['dim_content']}

    WHERE content_created_date IS NOT NULL
)

SELECT
    p.client_hash_id,
    p.content_hash_id,
    p.previous_30d_impressions,
    p.previous_30d_clicks,
    p.previous_30d_avg_position,
    q.visible_query_count,
    q.top_query_share,
    a.age_days

FROM previous_window p

LEFT JOIN query_signals q
    ON p.content_hash_id = q.content_hash_id

LEFT JOIN content_age a
    ON p.content_hash_id = a.content_hash_id

WHERE p.previous_30d_impressions > 0
""").df()

print("Rows available for scoring:", len(baseline_data))
baseline_data.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows available for scoring: 153559


,client_hash_id,content_hash_id,previous_30d_impressions,previous_30d_clicks,previous_30d_avg_position,visible_query_count,top_query_share,age_days
0,client_08a6a72ff48e62c0,content_447b93d5ff670356,67.0,0.0,12.862778,2,0.708333,214
1,client_08a6a72ff48e62c0,content_4483e354a2992497,47.0,1.0,7.652506,13,0.308642,163
2,client_08a6a72ff48e62c0,content_4489a53aa451dca0,3.0,0.0,24.666667,4,0.500000,213
3,client_08a6a72ff48e62c0,content_44afc7eca6a713b7,3.0,0.0,2.333333,1,1.000000,214
4,client_08a6a72ff48e62c0,content_44b4b22cfcc297c5,4.0,0.0,10.500000,6,0.231250,298


OSError: Cannot save file into a non-existent directory: 'work/outputs'

In [ ]:
# Create a simple baseline score

baseline_data["score"] = 0

# Meaningful search visibility
baseline_data.loc[
    baseline_data["previous_30d_impressions"] >= 100,
    "score"
] += 40

# Older content gets supporting points
baseline_data.loc[
    baseline_data["age_days"] >= 365,
    "score"
] += 30

baseline_data.loc[
    (baseline_data["age_days"] >= 180) &
    (baseline_data["age_days"] < 365),
    "score"
] += 15

# Pages with a reasonable search position get supporting points
baseline_data.loc[
    baseline_data["previous_30d_avg_position"] <= 20,
    "score"
] += 20

# More visible queries provide additional context
baseline_data.loc[
    baseline_data["visible_query_count"] >= 5,
    "score"
] += 10

# Assign one reason code and action
def assign_reason(row):
    if (
        row["previous_30d_impressions"] >= 100
        and row["age_days"] >= 365
    ):
        return "STALE_VISIBLE"

    elif (
        row["previous_30d_impressions"] >= 100
        and row["previous_30d_avg_position"] <= 20
    ):
        return "VISIBLE_OPPORTUNITY"

    else:
        return "REVIEW"


baseline_data["reason_code"] = baseline_data.apply(assign_reason, axis=1)

baseline_data["action"] = baseline_data["reason_code"].map(actions)

# Rank highest score first
baseline_data = baseline_data.sort_values(
    ["score", "previous_30d_impressions"],
    ascending=[False, False]
).reset_index(drop=True)

baseline_data["rank"] = baseline_data.index + 1

# Save the ranked queue
output_cols = [
    "rank",
    "client_hash_id",
    "content_hash_id",
    "score",
    "reason_code",
    "action",
    "previous_30d_impressions",
    "previous_30d_clicks",
    "previous_30d_avg_position",
    "visible_query_count",
    "top_query_share",
    "age_days"
]

baseline_queue = baseline_data[output_cols]

# Create the required output folder
import os
os.makedirs("work/outputs", exist_ok=True)

# Save the ranked queue
baseline_queue.to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)

print("CSV saved successfully.")
print("Rows ranked:", len(baseline_queue))


print("Top 10:")
baseline_queue.head(10)

CSV saved successfully.
Rows ranked: 153559
Rows ranked: 153559
Top 10:


,rank,client_hash_id,content_hash_id,score,reason_code,action,previous_30d_impressions,previous_30d_clicks,previous_30d_avg_position,visible_query_count,top_query_share,age_days
0,1,client_73cda7b4e4f265ea,content_8e1334d6356668e3,100,STALE_VISIBLE,REFRESH_REVIEW,203401.0,2.0,4.967059,47,0.982780,380
1,2,client_73cda7b4e4f265ea,content_fec55986a1868d62,100,STALE_VISIBLE,REFRESH_REVIEW,193954.0,0.0,3.844678,33,0.980960,380
2,3,client_73cda7b4e4f265ea,content_e241d6415ac9e534,100,STALE_VISIBLE,REFRESH_REVIEW,164152.0,401.0,2.925926,639,0.099182,382
3,4,client_73cda7b4e4f265ea,content_00d4fdf6e48a2d38,100,STALE_VISIBLE,REFRESH_REVIEW,129333.0,626.0,5.407108,997,0.101407,380
4,5,client_73cda7b4e4f265ea,content_c9f840183215651b,100,STALE_VISIBLE,REFRESH_REVIEW,125035.0,0.0,9.366950,26,0.150895,380
5,6,client_73cda7b4e4f265ea,content_cf651123f1085418,100,STALE_VISIBLE,REFRESH_REVIEW,123708.0,307.0,5.743877,535,0.049896,382
6,7,client_e547b89c05043229,content_ec2e0346994fb5a5,100,STALE_VISIBLE,REFRESH_REVIEW,119854.0,490.0,2.594817,608,0.043455,404
7,8,client_73cda7b4e4f265ea,content_fd2117c2c6790e4b,100,STALE_VISIBLE,REFRESH_REVIEW,105380.0,315.0,3.532104,629,0.365968,380
8,9,client_73cda7b4e4f265ea,content_b17c1d1cb0a346d6,100,STALE_VISIBLE,REFRESH_REVIEW,84169.0,378.0,4.434889,381,0.166035,366
9,10,client_73cda7b4e4f265ea,content_d508c9c6173af446,100,STALE_VISIBLE,REFRESH_REVIEW,77248.0,183.0,4.526262,352,0.084255,382


In [ ]:
import os

print("File exists:", os.path.exists("work/outputs/baseline_action_score.csv"))

File exists: True


In [ ]:
baseline_queue.head(20)

,rank,client_hash_id,content_hash_id,score,reason_code,action,previous_30d_impressions,previous_30d_clicks,previous_30d_avg_position,visible_query_count,top_query_share,age_days
0,1,client_73cda7b4e4f265ea,content_8e1334d6356668e3,100,STALE_VISIBLE,REFRESH_REVIEW,203401.0,2.0,4.967059,47,0.982780,380
1,2,client_73cda7b4e4f265ea,content_fec55986a1868d62,100,STALE_VISIBLE,REFRESH_REVIEW,193954.0,0.0,3.844678,33,0.980960,380
2,3,client_73cda7b4e4f265ea,content_e241d6415ac9e534,100,STALE_VISIBLE,REFRESH_REVIEW,164152.0,401.0,2.925926,639,0.099182,382
3,4,client_73cda7b4e4f265ea,content_00d4fdf6e48a2d38,100,STALE_VISIBLE,REFRESH_REVIEW,129333.0,626.0,5.407108,997,0.101407,380
4,5,client_73cda7b4e4f265ea,content_c9f840183215651b,100,STALE_VISIBLE,REFRESH_REVIEW,125035.0,0.0,9.366950,26,0.150895,380
5,6,client_73cda7b4e4f265ea,content_cf651123f1085418,100,STALE_VISIBLE,REFRESH_REVIEW,123708.0,307.0,5.743877,535,0.049896,382
6,7,client_e547b89c05043229,content_ec2e0346994fb5a5,100,STALE_VISIBLE,REFRESH_REVIEW,119854.0,490.0,2.594817,608,0.043455,404
7,8,client_73cda7b4e4f265ea,content_fd2117c2c6790e4b,100,STALE_VISIBLE,REFRESH_REVIEW,105380.0,315.0,3.532104,629,0.365968,380
8,9,client_73cda7b4e4f265ea,content_b17c1d1cb0a346d6,100,STALE_VISIBLE,REFRESH_REVIEW,84169.0,378.0,4.434889,381,0.166035,366
9,10,client_73cda7b4e4f265ea,content_d508c9c6173af446,100,STALE_VISIBLE,REFRESH_REVIEW,77248.0,183.0,4.526262,352,0.084255,382


## 3. Top-20 review

I reviewed the top 20 items from the baseline queue. Most of the highest-ranked items receive the STALE_VISIBLE reason because they have meaningful search visibility and are more than one year old.

The recommendations are decision-support rather than automatic refresh instructions. The confidence is moderate because the rule is based on observed signals and the ML-06 audit showed that staleness has only a mixed relationship with impression decline. A recommendation could be wrong if the page is intentionally evergreen, still performs well, has seasonal demand, or does not actually need a content update.

In [ ]:
# Create the top-20 review table

top20 = baseline_queue.head(20).copy()

def confidence_note(row):
    if (
        row["previous_30d_impressions"] >= 100
        and row["age_days"] >= 365
    ):
        return "Moderate: strong visibility and age support review, but age alone does not prove a refresh is needed."
    else:
        return "Low to moderate: signals suggest review, but evidence is not strong enough for an automatic action."


def what_would_make_it_wrong(row):
    return (
        "The recommendation could be wrong if the content is intentionally evergreen, "
        "still performs well, demand is seasonal, or the page does not need updating."
    )


top20["confidence_note"] = top20.apply(
    confidence_note,
    axis=1
)

top20["what_would_make_it_wrong"] = top20.apply(
    what_would_make_it_wrong,
    axis=1
)

review_cols = [
    "rank",
    "content_hash_id",
    "score",
    "action",
    "reason_code",
    "confidence_note",
    "what_would_make_it_wrong"
]

top20_review = top20[review_cols]

top20_review

,rank,content_hash_id,score,action,reason_code,confidence_note,what_would_make_it_wrong
0,1,content_8e1334d6356668e3,100,REFRESH_REVIEW,STALE_VISIBLE,Moderate: strong visibility and age support re...,The recommendation could be wrong if the conte...
1,2,content_fec55986a1868d62,100,REFRESH_REVIEW,STALE_VISIBLE,Moderate: strong visibility and age support re...,The recommendation could be wrong if the conte...
2,3,content_e241d6415ac9e534,100,REFRESH_REVIEW,STALE_VISIBLE,Moderate: strong visibility and age support re...,The recommendation could be wrong if the conte...
3,4,content_00d4fdf6e48a2d38,100,REFRESH_REVIEW,STALE_VISIBLE,Moderate: strong visibility and age support re...,The recommendation could be wrong if the conte...
4,5,content_c9f840183215651b,100,REFRESH_REVIEW,STALE_VISIBLE,Moderate: strong visibility and age support re...,The recommendation could be wrong if the conte...
5,6,content_cf651123f1085418,100,REFRESH_REVIEW,STALE_VISIBLE,Moderate: strong visibility and age support re...,The recommendation could be wrong if the conte...
6,7,content_ec2e0346994fb5a5,100,REFRESH_REVIEW,STALE_VISIBLE,Moderate: strong visibility and age support re...,The recommendation could be wrong if the conte...
7,8,content_fd2117c2c6790e4b,100,REFRESH_REVIEW,STALE_VISIBLE,Moderate: strong visibility and age support re...,The recommendation could be wrong if the conte...
8,9,content_b17c1d1cb0a346d6,100,REFRESH_REVIEW,STALE_VISIBLE,Moderate: strong visibility and age support re...,The recommendation could be wrong if the conte...
9,10,content_d508c9c6173af446,100,REFRESH_REVIEW,STALE_VISIBLE,Moderate: strong visibility and age support re...,The recommendation could be wrong if the conte...


### Review observation

The top 20 items all received the maximum score of 100 and the same STALE_VISIBLE reason code. This means the baseline has a saturated top tier rather than producing fine-grained score differences. The ordering among these tied items is therefore mainly determined by previous 30-day impressions. This is a limitation of the simple baseline and is something a later ML model could potentially improve.

## 4. Weak picks + leakage check

The main weak point in the baseline is that the top-ranked items are concentrated in the same STALE_VISIBLE category and many receive the maximum score. This makes the ranking less fine-grained than it could be.

I also checked the inputs used by the rule. The score uses previous-window performance, content age, search position, and query-level signals. It does not use the March outcome or a label-derived field, so the baseline does not intentionally use future outcome information.

In [ ]:
# Weak-pick review and leakage check

print("=== Weak-pick review ===")

# Show the lowest-scoring items that still entered the queue
weak_picks = baseline_queue.tail(10)[
    [
        "rank",
        "content_hash_id",
        "score",
        "reason_code",
        "action",
        "previous_30d_impressions",
        "previous_30d_avg_position",
        "age_days"
    ]
]

weak_picks

=== Weak-pick review ===


,rank,content_hash_id,score,reason_code,action,previous_30d_impressions,previous_30d_avg_position,age_days
153549,153550,content_c1ab0ba92f148dd8,0,REVIEW,GENERAL_REVIEW,1.0,82.0,131
153550,153551,content_ba7b93bc64b3e9cf,0,REVIEW,GENERAL_REVIEW,1.0,29.0,150
153551,153552,content_3e38e3c6d54a6c72,0,REVIEW,GENERAL_REVIEW,1.0,40.0,83
153552,153553,content_845eace4b9e41550,0,REVIEW,GENERAL_REVIEW,1.0,76.0,170
153553,153554,content_ac58086f7f1ae256,0,REVIEW,GENERAL_REVIEW,1.0,41.0,169
153554,153555,content_424b364b26f9fced,0,REVIEW,GENERAL_REVIEW,1.0,36.0,83
153555,153556,content_61b19a81d6150414,0,REVIEW,GENERAL_REVIEW,1.0,41.0,80
153556,153557,content_ac8607f50c3fda94,0,REVIEW,GENERAL_REVIEW,1.0,55.0,170
153557,153558,content_cb9686c9042f699b,0,REVIEW,GENERAL_REVIEW,1.0,24.0,82
153558,153559,content_32005e24fcee96fa,0,REVIEW,GENERAL_REVIEW,1.0,31.0,170


In [ ]:
print("\n=== Leakage check ===")

# Fields actually used by the baseline score
baseline_inputs = [
    "previous_30d_impressions",
    "previous_30d_clicks",
    "previous_30d_avg_position",
    "visible_query_count",
    "top_query_share",
    "age_days"
]

print("Baseline input fields:")
for field in baseline_inputs:
    print("-", field)

# Check for obvious future/label-derived fields
future_or_label_terms = [
    "decline",
    "label",
    "future",
    "outcome",
    "march_impressions",
    "is_declining"
]

found_leakage = [
    field for field in baseline_inputs
    if any(term in field.lower() for term in future_or_label_terms)
]

print("\nPotential future/label-derived inputs:", found_leakage)

if not found_leakage:
    print("LEAKAGE CHECK: PASS")
else:
    print("LEAKAGE CHECK: REVIEW REQUIRED")


=== Leakage check ===
Baseline input fields:
- previous_30d_impressions
- previous_30d_clicks
- previous_30d_avg_position
- visible_query_count
- top_query_share
- age_days

Potential future/label-derived inputs: []
LEAKAGE CHECK: PASS


## Self-check

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done